In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost
import divisi

In [2]:
# Load data
df_train = pd.read_csv("train-airline.csv")
df_test = pd.read_csv("test-airline.csv")
df_all = pd.concat([df_train, df_test])
df_prepped = df_all.drop(columns=['index', 'id']) #.sample(n=5000, random_state=1234)

In [3]:
# Preprocess data
def map_to_category(value):
    if value == 0:
        return "not applicable"
    elif value == 1:
        return "not satisfied"
    elif 2 <= value <= 4:
        return "neutral"
    elif value == 5:
        return "very satisfied"
    else:
        return "unknown" 
        
columns_5 = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]

df_prepped_cat = df_prepped.copy()

for column in columns_5:
    df_prepped_cat[column] = df_prepped_cat[column].apply(map_to_category)

In [4]:
# Train an XGBoost model to predict dissatisfaction

import xgboost as xgb
y = df_prepped_cat['satisfaction'] == 'satisfied'

X = df_prepped_cat.drop(columns=['satisfaction'])

X_continuous = X[['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']]
X_categorical = X.drop(columns=['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes'])

X_encoded = pd.get_dummies(X_categorical)
X = pd.concat([X_continuous, X_encoded], axis=1)

X = X.fillna(-999)  # Or use X.fillna(X.mean()) to fill with column means

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=1234)

# Initialize and train the XGBoost model
model = xgb.XGBClassifier(random_state=1234)
model.fit(X_train, y_train)

# Make predictions
model_preds = model.predict(X)
model_probs = model.predict_proba(X)
is_error = model_preds != y

print(f"Train + test error rate: {is_error.mean():.2%}")

Train + test error rate: 4.84%


Discretize the dataset so that we can perform subgroup analysis. Divisi chooses sensible defaults for the dataframe you provide, and you can specify columns to remove and customize.


In [5]:
discrete_df = divisi.discretize_data(df_prepped_cat, custom_cols={
    'Age': divisi.bin_values(quantiles=5),
    'Flight Distance': divisi.bin_values(bins=[500, 1000, 2000, 10000, 130000]),
    'Departure Delay in Minutes': divisi.bin_values(bins=[0, 15, 30, 60, 1000]), 
    'Arrival Delay in Minutes': divisi.bin_values(bins=[0, 15, 30, 60, 1000]), 
}, remove_cols=["satisfaction"])
discrete_df

<DiscretizedData:
	Gender: 0 - 'Female' (50.74%), 1 - 'Male' (49.26%)
	Customer Type: 0 - 'Loyal Customer' (81.69%), 1 - 'disloyal Customer' (18.31%)
	Age: 0 - '< 25' (18.99%), 1 - '25 - 36' (20.85%), 2 - '36 - 44' (18.92%), 3 - '44 - 54' (21.21%), 4 - '> 54' (20.03%)
	Type of Travel: 0 - 'Business travel' (69.06%), 1 - 'Personal Travel' (30.94%)
	Class: 0 - 'Business' (47.86%), 1 - 'Eco' (44.89%), 2 - 'Eco Plus' (7.25%)
	Flight Distance: 0 - '< 500' (30.85%), 1 - '500 - 1000' (26.96%), 2 - '1000 - 2000' (21.49%), 3 - '2000 - 10000' (20.70%), 4 - '10000 - 1.3e+05' (0.00%), 5 - '> 1.3e+05' (0.00%)
	Inflight wifi service: 0 - 'neutral' (68.74%), 1 - 'not applicable' (3.02%), 2 - 'not satisfied' (17.19%), 3 - 'very satisfied' (11.05%)
	Departure/Arrival time convenient: 0 - 'neutral' (58.36%), 1 - 'not applicable' (5.14%), 2 - 'not satisfied' (14.94%), 3 - 'very satisfied' (21.56%)
	Ease of Online booking: 0 - 'neutral' (65.36%), 1 - 'not applicable' (4.37%), 2 - 'not satisfied' (16.85%),

In [ ]:
w = divisi.DivisiWidget(
    discrete_df, 
    state_path="divisi_state", # provide a path to store interface state so you can pick up where you left off
    # metrics to display for each subgroup
    metrics={
        "Dissatisfaction": 1 - y.values,
        "Model Probability": model_probs[:,1],
        "Error": is_error.values
    })
w

# Programmatic Method

Use this to discover subgroups without using the interactive interface.

In [7]:
finder = divisi.sampling.SamplingSubgroupSearch(
    discrete_df,
    {
        # Specify ranking functions manually
        "Dissatisfaction Rate": divisi.ranking.OutcomeRate(1 - y.values),
        "Dissatisfaction Interaction Effect": divisi.ranking.InteractionEffect(1 - y.values),
        "Simple Rule": divisi.ranking.SimpleRule()
    },
    # additional sampling options
    min_items_fraction=0.05
)
results, _ = finder.sample(50)
for rule in results.rank({"Dissatisfaction Rate": 1.0, "Dissatisfaction Interaction Effect": 1.0, "Simple Rule": 0.25}):
    mask = discrete_df.mask_for_rule(rule)
    print(f"{rule}: Count = {mask.sum()}, Dissatisfaction Rate = {(1 - y.values)[discrete_df.mask_for_rule(rule)].mean():.2%}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:08<00:00,  5.98it/s]


<Slice: (Inflight wifi service = not satisfied & Gate location = neutral)>: Count = 11268, Dissatisfaction Rate = 99.97%
<Slice: (Inflight wifi service = not satisfied & Departure/Arrival time convenient = neutral)>: Count = 7531, Dissatisfaction Rate = 97.85%
<Slice: (Ease of Online booking = not satisfied & Gate location = neutral)>: Count = 8967, Dissatisfaction Rate = 92.77%
<Slice: (Gate location = neutral & Departure/Arrival time convenient = not satisfied)>: Count = 6577, Dissatisfaction Rate = 87.18%
<Slice: (Class = Eco & Inflight wifi service = not satisfied)>: Count = 9572, Dissatisfaction Rate = 99.98%
<Slice: (Inflight wifi service = neutral & Departure/Arrival time convenient = very satisfied)>: Count = 14749, Dissatisfaction Rate = 86.94%
<Slice: (Type of Travel = Personal Travel & Inflight wifi service = not satisfied)>: Count = 7298, Dissatisfaction Rate = 100.00%
<Slice: (Online boarding = not satisfied & Inflight wifi service = not satisfied)>: Count = 9831, Dissatis